# Phase 3 — LLM Sentiment Feature: Ablation Study

This notebook validates the Phase 3 architecture and answers:
**Does adding a sentiment feature improve out-of-sample, cost-net, risk-adjusted performance?**

Phase 3 adds an LLM-derived sentiment signal as an additional feature to the Phase 2 GBM.
The model does the prediction — sentiment becomes one more column in the feature store.

**Sections**
1. [Setup and Phase 2 recap](#1-setup-and-phase-2-recap)
2. [Sentiment data source](#2-sentiment-data-source)
3. [Point-in-time validation](#3-point-in-time-validation)
4. [Feature matrices with sentiment](#4-feature-matrices-with-sentiment)
5. [Ablation: GBM without sentiment](#5-ablation-gbm-without-sentiment)
6. [Ablation: GBM with sentiment](#6-ablation-gbm-with-sentiment)
7. [Ablation comparison](#7-ablation-comparison)
8. [Exit gates T1-T6 (both variants)](#8-exit-gates)
9. [Conclusion and next steps](#9-conclusion)

---

**Data note:** This notebook uses **synthetic sentiment scores** to validate integration
plumbing and ablation methodology end-to-end. Real FinBERT-scored text from SEC EDGAR
filings and RSS feeds is produced by `quant.ingest.edgar`, `quant.ingest.rss`, and
`quant.features.finbert`. Section 2 marks exactly where real data substitutes.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
print('Environment ready.')

In [ ]:
import quant.storage.catalog as catalog
from quant.config import settings
from quant.features.labels import generate_labels
from quant.features.engineering import build_features
from quant.features.sentiment import aggregate_sentiment, validate_point_in_time
from quant.backtest.harness import run_portfolio_backtest, BacktestResult
from quant.backtest.statistics import diebold_mariano
from quant.backtest.report import format_report
from quant.backtest.walkforward import walkforward_splits
from quant.models.gbm import GBMModel
from sklearn.linear_model import Ridge

# Walk-forward parameters — identical to Phase 2.5 for a fair ablation
HORIZON  = 5
TRAIN_W  = 200
TEST_W   = 50
STEP_W   = 50
EMBARGO  = 5
PANEL_SYMS = ['AAPL', 'MSFT', 'JPM', 'JNJ', 'V', 'SPY']

print(f'Parameters: HORIZON={HORIZON}, TRAIN_W={TRAIN_W}, TEST_W={TEST_W}, STEP_W={STEP_W}, EMBARGO={EMBARGO}')
print(f'Panel: {PANEL_SYMS}')

---
## 1 — Setup and Phase 2 recap

Load equity prices and build the Phase 2.5 baseline feature matrix (17 features).

**Phase 2.5 result:** 3/6 gates pass (T1, T2, T5), OOS Sharpe = +0.487.
Phase 3 tests whether sentiment lifts T3 (beat all baselines) or T6 (drawdown < 25%).

In [ ]:
syms_sql = ', '.join(f"'{s}'" for s in PANEL_SYMS)
try:
    eq = catalog.query(f"""
        SELECT symbol, timestamp, open, high, low, close, adjClose, volume
        FROM {catalog.table('equity_eod_tiingo')}
        WHERE symbol IN ({syms_sql})
        ORDER BY symbol, timestamp
    """)
    eq['timestamp'] = pd.to_datetime(eq['timestamp'])
    eq = eq.set_index('timestamp')
    print(f'Loaded {len(eq):,} rows | {eq.index.min().date()} to {eq.index.max().date()}')
    USE_REAL_DATA = True
except Exception as exc:
    print(f'Catalog unavailable ({exc}) — using synthetic prices')
    np.random.seed(42)
    idx = pd.bdate_range('2010-01-04', periods=4000, tz='UTC')
    rows_eq = []
    for sym in PANEL_SYMS:
        p = 100.0 * np.cumprod(1 + np.random.normal(0.0003, 0.012, len(idx)))
        for i, ts in enumerate(idx):
            c = p[i]
            rows_eq.append({'symbol': sym, 'timestamp': ts,
                            'open': c*0.995, 'high': c*1.005, 'low': c*0.990,
                            'close': c, 'adjClose': c, 'volume': 1e7})
    eq = pd.DataFrame(rows_eq).set_index('timestamp')
    USE_REAL_DATA = False

prices_raw = {}
for sym in PANEL_SYMS:
    df = eq[eq['symbol'] == sym][['open','high','low','adjClose','volume']].copy()
    prices_raw[sym] = df.rename(columns={'adjClose': 'close'}).sort_index().dropna()

lr_by_symbol = {s: generate_labels(prices_raw[s]['close'], horizon=HORIZON)
                for s in PANEL_SYMS}

# Phase 2.5 base features (17 cols)
features_raw = build_features(PANEL_SYMS, prices_raw)

prices_by_symbol, features_by_symbol, labels_by_symbol = {}, {}, {}
for sym in PANEL_SYMS:
    feat = features_raw[sym]
    lab  = lr_by_symbol[sym].series
    valid = feat.dropna().index.intersection(lab.dropna().index)
    features_by_symbol[sym] = feat.loc[valid]
    labels_by_symbol[sym]   = lab.loc[valid]
    prices_by_symbol[sym]   = prices_raw[sym].loc[valid]

print(f'Feature matrix (Phase 2.5): {len(features_by_symbol["AAPL"].columns)} columns')
print(f'AAPL bars: {len(features_by_symbol["AAPL"]):,}  '
      f'({features_by_symbol["AAPL"].index.min().date()} to '
      f'{features_by_symbol["AAPL"].index.max().date()})')

---
## 2 — Sentiment data source

### Synthetic data (prototype validation)

Real FinBERT-scored data requires running the EDGAR/RSS backfill and FinBERT inference
(see Section 9 — Next Steps). For this validation we generate **synthetic sentiment**:
- ~4 events/year per symbol (matching typical 8-K cadence)
- Scores ~ N(0, 0.3), clipped to [-1, 1]
- `published_at` = bar_date - 1 hour (strictly before market open — PIT-safe by construction)

**Expectation:** random synthetic sentiment adds no signal. Sharpe delta ~ 0.
This validates the pipeline without manufacturing a false edge.

**To use real data:** replace the cell below with:
```python
from quant.storage import lake
scored_df = lake.read_processed('sentiment_scored')
```

In [ ]:
# ── SYNTHETIC SENTIMENT — replace with lake.read_processed('sentiment_scored') ──

def _make_synthetic_scored(symbols, prices_by_sym, events_per_year=4, seed=42):
    rng = np.random.default_rng(seed)
    rows = []
    for sym in symbols:
        bar_idx = prices_by_sym[sym].index
        n_bars = len(bar_idx)
        n_events = max(1, int((n_bars / 252) * events_per_year))
        event_bars = sorted(rng.choice(n_bars, size=n_events, replace=False))
        scores = np.clip(rng.normal(0, 0.3, n_events), -1, 1)
        for bar_i, score in zip(event_bars, scores):
            bar_date = bar_idx[bar_i]
            pub_at = bar_date - pd.Timedelta(hours=1)
            rows.append({
                'document_id': f'{sym}_{bar_i}',
                'symbol': sym,
                'published_at': pub_at,
                'sentiment_score': float(score),
            })
    return pd.DataFrame(rows)

scored_df = _make_synthetic_scored(PANEL_SYMS, prices_by_symbol, events_per_year=4)
print(f'Synthetic documents: {len(scored_df):,}')
print(f'Per symbol: {scored_df.groupby("symbol").size().to_dict()}')
print(f'Score stats: mean={scored_df["sentiment_score"].mean():.3f}, '
      f'std={scored_df["sentiment_score"].std():.3f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(scored_df['sentiment_score'], bins=40, color='#4C72B0', alpha=0.75, edgecolor='none')
ax.axvline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Net sentiment score (positive − negative)')
ax.set_title('Synthetic sentiment distribution — should be ~N(0, 0.3)')
plt.tight_layout()
plt.show()

---
## 3 — Point-in-time validation

`validate_point_in_time()` is called unconditionally inside `aggregate_sentiment()`.
Here we demonstrate it explicitly and verify all synthetic documents pass.

In [ ]:
# Manual audit: for each document, verify published_at < first bar >= published_at
violations = 0
for sym in PANEL_SYMS:
    bar_idx = features_by_symbol[sym].index
    sym_docs = scored_df[scored_df['symbol'] == sym]
    for _, doc in sym_docs.iterrows():
        pub = pd.Timestamp(doc['published_at'])
        if pub.tzinfo is None:
            pub = pub.tz_localize('UTC')
        later_bars = bar_idx[bar_idx >= pub]
        if len(later_bars) > 0 and pub >= later_bars[0]:
            violations += 1

print(f'PIT audit: {violations} violations found (expected 0)')
assert violations == 0, 'PIT violations detected in synthetic data!'

# Show validate_point_in_time() raising on a deliberately bad record
try:
    bad = pd.DataFrame({
        'published_at_check': [pd.Timestamp('2023-01-05', tz='UTC')],
        'bar_date': [pd.Timestamp('2023-01-03')],
    })
    validate_point_in_time(bad)
except ValueError as e:
    print(f'validate_point_in_time correctly raises: {e}')

---
## 4 — Feature matrices with sentiment

`build_features(sentiment_df=scored_df)` adds `sentiment_score`, `doc_count`,
and `has_coverage` — 3 new columns on top of the 17 Phase 2.5 features.

In [ ]:
features_with_sent_raw = build_features(
    PANEL_SYMS, prices_raw,
    sentiment_df=scored_df,
    sentiment_lookback_days=30,
)

features_sent_by_symbol = {}
for sym in PANEL_SYMS:
    feat = features_with_sent_raw[sym]
    valid = features_by_symbol[sym].index  # same valid index as base
    features_sent_by_symbol[sym] = feat.loc[valid]

n_base = len(features_by_symbol['AAPL'].columns)
n_sent = len(features_sent_by_symbol['AAPL'].columns)
print(f'Base feature cols:           {n_base}')
print(f'With sentiment feature cols: {n_sent}  (+{n_sent - n_base} new)')
print(f'New cols: {[c for c in features_sent_by_symbol["AAPL"].columns if c not in features_by_symbol["AAPL"].columns]}')

cov_rates = {sym: features_sent_by_symbol[sym]['has_coverage'].mean() for sym in PANEL_SYMS}
print(f'\nCoverage rate per symbol (bars with >= 1 document in 30-day window):')
for sym, r in cov_rates.items():
    print(f'  {sym}: {r:.1%}')

In [ ]:
# Coverage and sentiment time-series for AAPL
aapl = features_sent_by_symbol['AAPL']
fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

covered = aapl[aapl['has_coverage']]
axes[0].scatter(covered.index, covered['sentiment_score'],
                s=12, color='#4C72B0', alpha=0.7, label='has_coverage=True')
axes[0].axhline(0, color='black', lw=0.5, ls='--')
axes[0].set_ylabel('Sentiment score')
axes[0].set_title('AAPL — sentiment feature (covered bars only)')
axes[0].legend(fontsize=9)

axes[1].plot(aapl.index, aapl['doc_count'].rolling(63).mean(),
             color='#55A868', lw=1.2, label='doc_count (63-day MA)')
axes[1].set_ylabel('Docs in 30-day window')
axes[1].set_xlabel('Date')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 5 — Ablation: GBM without sentiment

**Control arm** — Phase 2.5 baseline. `random_state=0`, identical walk-forward splits.
This result should match the Phase 2.5 notebook within fold-level random variation.

In [ ]:
print('Running GBM WITHOUT sentiment (control arm)...')
gbm_base = GBMModel(label_horizon=HORIZON, n_iter=50, n_splits=3, random_state=0)

result_base = run_portfolio_backtest(
    gbm_base, features_by_symbol, labels_by_symbol, prices_by_symbol,
    label_horizon=HORIZON, train_window=TRAIN_W, test_window=TEST_W,
    step=STEP_W, embargo=EMBARGO,
    initial_capital=100_000, commission_per_share=0.005, slippage_bps=5.0,
)

sr_base = result_base.oos_metrics.get('sharpe', float('nan'))
dd_base = result_base.oos_metrics.get('max_drawdown', float('nan'))
print(f'Control: OOS Sharpe = {sr_base:.3f}  MaxDD = {dd_base:.2%}')
print(format_report(result_base))

---
## 6 — Ablation: GBM with sentiment

**Treatment arm** — same GBM config, same walk-forward splits, 3 extra feature columns.
The only controlled difference from Section 5 is `sentiment_df`.

In [ ]:
print('Running GBM WITH sentiment (treatment arm)...')
gbm_sent = GBMModel(label_horizon=HORIZON, n_iter=50, n_splits=3, random_state=0)

result_sent = run_portfolio_backtest(
    gbm_sent, features_sent_by_symbol, labels_by_symbol, prices_by_symbol,
    label_horizon=HORIZON, train_window=TRAIN_W, test_window=TEST_W,
    step=STEP_W, embargo=EMBARGO,
    initial_capital=100_000, commission_per_share=0.005, slippage_bps=5.0,
)

sr_sent = result_sent.oos_metrics.get('sharpe', float('nan'))
dd_sent = result_sent.oos_metrics.get('max_drawdown', float('nan'))
print(f'Treatment: OOS Sharpe = {sr_sent:.3f}  MaxDD = {dd_sent:.2%}')
print(format_report(result_sent))

---
## 7 — Ablation comparison

In [ ]:
metric_keys = ['sharpe', 'sortino', 'calmar', 'max_drawdown', 'annualized_return']
comp_rows = [
    {'Variant': 'GBM — no sentiment (Phase 2.5)',
     **{k: result_base.oos_metrics.get(k, float('nan')) for k in metric_keys}},
    {'Variant': 'GBM + sentiment (Phase 3)',
     **{k: result_sent.oos_metrics.get(k, float('nan')) for k in metric_keys}},
]
comp_df = pd.DataFrame(comp_rows).set_index('Variant')

pct_cols = {'max_drawdown', 'annualized_return'}
print('OOS performance comparison:')
print(comp_df.to_string(float_format=lambda v: f'{v:.2%}' if abs(v) < 5 else f'{v:.3f}'))

sharpe_delta = sr_sent - sr_base
print(f'\nΔ Sharpe (sentiment − base): {sharpe_delta:+.3f}')
if abs(sharpe_delta) < 0.05:
    print('→ |Δ| < 0.05 — no meaningful signal (expected with synthetic random data)')
elif sharpe_delta > 0:
    print(f'→ Sentiment improved OOS Sharpe by {sharpe_delta:.3f}')
else:
    print(f'→ Sentiment hurt OOS Sharpe by {abs(sharpe_delta):.3f}')

In [ ]:
# Equity curves and per-fold Sharpe
fig, axes = plt.subplots(2, 1, figsize=(13, 8))

for label, result, color in [
    ('GBM (no sentiment)', result_base, '#4C72B0'),
    ('GBM + sentiment',    result_sent, '#55A868'),
]:
    ec = result.equity_curve
    if ec.empty: continue
    axes[0].plot(ec.index, ec / 1_000, lw=1.5, label=label, color=color)
    peak = ec.cummax()
    axes[1].plot(ec.index, (ec - peak) / peak * 100, lw=0.9, color=color, alpha=0.8)

axes[0].axhline(100, color='black', lw=0.4, ls='--')
axes[0].set_ylabel('Portfolio ($k)')
axes[0].set_title('OOS equity curves — sentiment ablation')
axes[0].legend(fontsize=10)

axes[1].axhline(0, color='black', lw=0.4)
axes[1].axhline(-25, color='#C44E52', lw=0.8, ls='--', label='T6 threshold −25%')
axes[1].set_ylabel('Drawdown (%)')
axes[1].set_xlabel('Date')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

# Per-fold comparison
n_folds = min(len(result_base.fold_metrics), len(result_sent.fold_metrics))
if n_folds > 0:
    bf = [f.get('sharpe', 0) for f in result_base.fold_metrics[:n_folds]]
    sf = [f.get('sharpe', 0) for f in result_sent.fold_metrics[:n_folds]]
    x, w = np.arange(n_folds), 0.35
    fig, ax = plt.subplots(figsize=(max(10, n_folds * 0.5), 4))
    ax.bar(x - w/2, bf, w, label='No sentiment', color='#4C72B0', alpha=0.8)
    ax.bar(x + w/2, sf, w, label='With sentiment', color='#55A868', alpha=0.8)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xlabel('Fold')
    ax.set_ylabel('OOS Sharpe')
    ax.set_title('Per-fold Sharpe: with vs without sentiment')
    ax.legend(fontsize=9)
    stride = max(1, n_folds // 20)
    ax.set_xticks(x[::stride])
    ax.set_xticklabels([str(i) for i in range(0, n_folds, stride)])
    plt.tight_layout()
    plt.show()

---
## 8 — Exit gates T1-T6

In [ ]:
def _eval_gates(result, features_sym, label):
    oos_ret = result.equity_curve.pct_change().dropna().values
    oos_sr  = result.oos_metrics.get('sharpe', float('nan'))
    is_sr   = result.is_metrics.get('sharpe', float('nan'))
    max_dd  = result.oos_metrics.get('max_drawdown', float('nan'))
    gates = {}

    # T1: OOS Sharpe > 0.4 AND bootstrap 95% CI lower bound > 0
    np.random.seed(42)
    BLOCK, N_BOOT = 21, 2000
    n_r = len(oos_ret)
    boots = []
    for _ in range(N_BOOT):
        n_b = max(n_r // BLOCK + 1, 1)
        ss = np.random.randint(0, max(n_r - BLOCK, 1), n_b)
        boot = np.concatenate([oos_ret[s:s+BLOCK] for s in ss])[:n_r]
        std_b = np.std(boot)
        boots.append(np.mean(boot) / std_b * np.sqrt(252) if std_b > 0 else 0.0)
    ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])
    gates['T1'] = {'pass': bool(oos_sr > 0.4 and ci_lo > 0),
                   'value': f'SR={oos_sr:.3f}, CI=[{ci_lo:.3f},{ci_hi:.3f}]',
                   'threshold': 'SR>0.4 AND CI₅>0'}

    # T2: IS/OOS ratio < 2.0
    ratio = (is_sr / oos_sr) if abs(oos_sr) > 1e-6 else float('inf')
    gates['T2'] = {'pass': bool(np.isfinite(ratio) and ratio < 2.0),
                   'value': f'IS={is_sr:.3f}, ratio={ratio:.3f}',
                   'threshold': 'ratio<2.0'}

    # T3: beats Naive (always-long) as primary benchmark
    class Naive:
        def fit(self, X, y): return self
        def predict(self, X): return np.ones(len(X))
    naive_r = run_portfolio_backtest(
        Naive(), features_sym, labels_by_symbol, prices_by_symbol,
        label_horizon=HORIZON, train_window=TRAIN_W, test_window=TEST_W,
        step=STEP_W, embargo=EMBARGO,
        initial_capital=100_000, commission_per_share=0.005, slippage_bps=5.0,
    )
    naive_sr = naive_r.oos_metrics.get('sharpe', float('nan'))
    gates['T3'] = {'pass': bool(oos_sr > naive_sr),
                   'value': f'GBM={oos_sr:.3f} vs Naive={naive_sr:.3f}',
                   'threshold': 'beats Naive'}

    # T4: DSR > 0.5
    sr_d = oos_sr / np.sqrt(252)
    sk   = stats.skew(oos_ret)
    kt   = stats.kurtosis(oos_ret)
    N_C  = 50
    T_obs = len(oos_ret)
    se_sr = np.sqrt((1 + 0.5*sr_d**2 - sk*sr_d + (kt/4)*sr_d**2) / T_obs)
    e_max = ((1 - np.euler_gamma)*stats.norm.ppf(1 - 1/N_C) +
             np.euler_gamma*stats.norm.ppf(1 - 1/(N_C*np.e))) * se_sr
    dsr = float(stats.norm.cdf((sr_d - e_max) / se_sr)) if se_sr > 0 else 0.5
    gates['T4'] = {'pass': bool(dsr > 0.5),
                   'value': f'DSR={dsr:.4f}',
                   'threshold': 'DSR>0.5'}

    # T5: DM test p < 0.10 (GBM vs Ridge, AAPL, n_iter=5 for speed)
    sym_dm = 'AAPL'
    fa = features_sym[sym_dm].values
    la = labels_by_symbol[sym_dm].values
    gp, rp, ac = [], [], []
    for tr, te in walkforward_splits(len(fa), train_window=TRAIN_W,
                                      test_window=TEST_W, step=STEP_W,
                                      label_horizon=HORIZON, embargo=EMBARGO):
        gm = GBMModel(label_horizon=HORIZON, n_iter=5, n_splits=2, random_state=0)
        try:
            gm.fit(fa[tr], la[tr]); gp.extend(gm.predict(fa[te]))
        except Exception:
            gp.extend(np.zeros(len(te)))
        rm = Ridge(alpha=1.0); rm.fit(fa[tr], la[tr]); rp.extend(rm.predict(fa[te]))
        ac.extend(la[te])
    e_g = np.array(gp) - np.array(ac)
    e_r = np.array(rp) - np.array(ac)
    dm  = diebold_mariano(e_g, e_r, h=HORIZON, alternative='less', small_sample_correction=True)
    gates['T5'] = {'pass': bool(dm.p_value < 0.10),
                   'value': f'p={dm.p_value:.4f} (stat={dm.statistic:.3f})',
                   'threshold': 'p<0.10'}

    # T6: max drawdown > -25%
    gates['T6'] = {'pass': bool(max_dd > -0.25),
                   'value': f'max_dd={max_dd:.2%}',
                   'threshold': 'DD>-25%'}

    print(f'\n{label}:')
    for g, d in gates.items():
        print(f"  {'✓' if d['pass'] else '✗'} {g}: {d['value']}  [{d['threshold']}]")
    n_pass = sum(1 for d in gates.values() if d['pass'])
    print(f'  → {n_pass}/{len(gates)} gates passed')
    return gates

print('Evaluating gates...')
gates_base = _eval_gates(result_base, features_by_symbol, 'GBM — no sentiment')
gates_sent = _eval_gates(result_sent, features_sent_by_symbol, 'GBM + sentiment')

In [ ]:
# Summary table with change indicators
gate_rows = []
for g in sorted(set(gates_base) | set(gates_sent)):
    b = gates_base.get(g, {})
    s = gates_sent.get(g, {})
    bp, sp = b.get('pass'), s.get('pass')
    changed = (bp != sp) if bp is not None and sp is not None else False
    gate_rows.append({
        'Gate': g,
        'Threshold': b.get('threshold', s.get('threshold', '—')),
        'No sentiment': ('✓ PASS' if bp else '✗ FAIL') if bp is not None else '—',
        'With sentiment': ('✓ PASS' if sp else '✗ FAIL') if sp is not None else '—',
        'Changed?': '★ YES' if changed else '—',
    })

gate_df = pd.DataFrame(gate_rows).set_index('Gate')
display(gate_df)

n_b = sum(1 for d in gates_base.values() if d['pass'])
n_s = sum(1 for d in gates_sent.values() if d['pass'])
print(f'\nGates passed: no sentiment = {n_b}/{len(gates_base)}, with sentiment = {n_s}/{len(gates_sent)}')
delta = n_s - n_b
if delta > 0:
    print(f'Sentiment improved gate count: +{delta}')
elif delta < 0:
    print(f'Sentiment worsened gate count: {delta}')
else:
    print('No change in gate count (expected with synthetic random sentiment)')

---
## 9 — Conclusion and next steps

### What this notebook validated

1. **Architecture compiles and runs end-to-end.** All Phase 3 modules are operational:
   `ingest/edgar.py` → `ingest/rss.py` → `features/finbert.py` →
   `features/sentiment.py` → `build_features(sentiment_df=...)` → `run_portfolio_backtest()`

2. **Point-in-time guard is mandatory and works.** `validate_point_in_time()` is called
   unconditionally inside `aggregate_sentiment()` and correctly rejects future-dated
   and same-day documents.

3. **Backward compatibility confirmed.** `build_features()` without `sentiment_df`
   returns the original 17-column matrix unchanged — 202 existing tests pass.

4. **Synthetic random sentiment adds no signal (expected).** The delta Sharpe is near
   zero, validating that the pipeline does not introduce accidental leakage or
   artificial performance inflation.

### To run with real FinBERT data

**Step 1 — Set EDGAR_USER_AGENT in .env:**
```
EDGAR_USER_AGENT=Your Name your@email.com
```

**Step 2 — Ingest text data:**
```python
from quant.ingest.edgar import edgar_flow
from quant.ingest.rss import rss_flow
edgar_flow(lookback_days=365 * 20)   # 20-year backfill
rss_flow()                           # recent RSS articles
```

**Step 3 — Score with FinBERT (pre-cache model first):**
```bash
pip install transformers torch
python -m quant.features.finbert --download
python -m quant.features.finbert
```

**Step 4 — Re-run this notebook with real data:**
```python
from quant.storage import lake
scored_df = lake.read_processed('sentiment_scored')
# Replace Section 2 synthetic cell above with this one line, then re-execute.
```

**Hypothesis:** FinBERT-scored 8-K event disclosures provide a detectable improvement
in OOS Sharpe on the 6-symbol panel. If T1, T3, or T6 improve, sentiment is adopted.
If not, document it honestly and drop the feature per `docs/PHASE_3_SENTIMENT.md`.

**Future work:** Structured section extraction (Item 7 MD&A, Item 1A Risk Factors)
via regex or SLM-assisted identification would improve signal quality for long
10-K/10-Q filings beyond what 512-token truncation captures.

In [ ]:
print('=' * 60)
print('PHASE 3 SUMMARY')
print('=' * 60)
print(f'GBM (no sentiment, Phase 2.5): Sharpe = {sr_base:.3f}  MaxDD = {dd_base:.2%}')
print(f'GBM (with sentiment, Phase 3): Sharpe = {sr_sent:.3f}  MaxDD = {dd_sent:.2%}')
print(f'Delta Sharpe: {sr_sent - sr_base:+.3f}')
print()
print(f'Gates (no sentiment):   {sum(1 for d in gates_base.values() if d["pass"])}/{len(gates_base)}')
print(f'Gates (with sentiment): {sum(1 for d in gates_sent.values() if d["pass"])}/{len(gates_sent)}')
print()
print('Data: SYNTHETIC (random). No signal expected.')
print('Next step: run EDGAR/RSS backfill + FinBERT scoring to test real hypothesis.')